In [8]:
import torch
torch.cuda.empty_cache()

In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.preprocessing import preprocess_test, preprocess_train, postprocess_test
from pypots.imputation import SAITS
from pypots.utils.metrics import calc_mae
from pypots.nn.modules.loss import MAE
import random

path = "saits/small_saits_X_test_sma.pypots"
saits = SAITS(n_steps=1057, n_features=9, n_layers=1, d_model=64, d_ffn=128, n_heads=2, d_k=32, d_v=32, dropout=0.1, patience=3, epochs=500, batch_size=64, saving_path=path, model_saving_strategy="best")
data = pd.read_csv("data/processed/mini_X_train.csv")
data_test = pd.read_csv('data/processed/y_train_holed.csv')
  

2025-05-24 19:06:36 [WARNING]: ‼️ `pypots.utils.metrics` is deprecated. Please import from `pypots.nn.functional` instead.
2025-05-24 19:06:36 [INFO]: No given device, using default device: cuda
2025-05-24 19:06:36 [INFO]: Model files will be saved to saits/small_saits_X_test_sma.pypots/20250524_T190636
2025-05-24 19:06:36 [INFO]: Tensorboard file will be saved to saits/small_saits_X_test_sma.pypots/20250524_T190636/tensorboard
2025-05-24 19:06:36 [INFO]: Using customized MAE as the training loss function.
2025-05-24 19:06:36 [INFO]: Using customized MSE as the validation metric function.
2025-05-24 19:06:36 [INFO]: SAITS initialized with the given hyperparameters, the number of trainable parameters: 79,727



████████╗██╗███╗   ███╗███████╗    ███████╗███████╗██████╗ ██╗███████╗███████╗    █████╗ ██╗
╚══██╔══╝██║████╗ ████║██╔════╝    ██╔════╝██╔════╝██╔══██╗██║██╔════╝██╔════╝   ██╔══██╗██║
   ██║   ██║██╔████╔██║█████╗█████╗███████╗█████╗  ██████╔╝██║█████╗  ███████╗   ███████║██║
   ██║   ██║██║╚██╔╝██║██╔══╝╚════╝╚════██║██╔══╝  ██╔══██╗██║██╔══╝  ╚════██║   ██╔══██║██║
   ██║   ██║██║ ╚═╝ ██║███████╗    ███████║███████╗██║  ██║██║███████╗███████║██╗██║  ██║██║
   ╚═╝   ╚═╝╚═╝     ╚═╝╚══════╝    ╚══════╝╚══════╝╚═╝  ╚═╝╚═╝╚══════╝╚══════╝╚═╝╚═╝  ╚═╝╚═╝
ai4ts v0.0.3 - building AI for unified time-series analysis, https://time-series.ai 



### Preprocess des données à entrainer

In [2]:
X = preprocess_train(data)
X_ori = X.copy()
#X = mcar(X, 0.1)
dataset = {"X": X}

### Entrainement du modèle + Sauvegarde

In [ ]:
saits.fit(dataset)
imputation = saits.impute(dataset)
indicating_mask = np.isnan(X) ^ np.isnan(X_ori)
mae = calc_mae(imputation, np.nan_to_num(X_ori), indicating_mask)
print(f"Le MAE sur le train set : {mae}")

2025-05-24 14:50:35 [INFO]: Epoch 001 - training loss (MAE): 1.2579
2025-05-24 14:51:29 [INFO]: Epoch 002 - training loss (MAE): 0.4901
2025-05-24 14:52:24 [INFO]: Epoch 003 - training loss (MAE): 0.4228
2025-05-24 14:53:19 [INFO]: Epoch 004 - training loss (MAE): 0.3928
2025-05-24 14:54:13 [INFO]: Epoch 005 - training loss (MAE): 0.3722
2025-05-24 14:55:08 [INFO]: Epoch 006 - training loss (MAE): 0.3358
2025-05-24 14:56:03 [INFO]: Epoch 007 - training loss (MAE): 0.2868
2025-05-24 14:56:57 [INFO]: Epoch 008 - training loss (MAE): 0.2638
2025-05-24 14:57:52 [INFO]: Epoch 009 - training loss (MAE): 0.2502
2025-05-24 14:58:47 [INFO]: Epoch 010 - training loss (MAE): 0.2396
2025-05-24 14:59:41 [INFO]: Epoch 011 - training loss (MAE): 0.2303
2025-05-24 15:00:36 [INFO]: Epoch 012 - training loss (MAE): 0.2229
2025-05-24 15:01:31 [INFO]: Epoch 013 - training loss (MAE): 0.2167
2025-05-24 15:02:25 [INFO]: Epoch 014 - training loss (MAE): 0.2114
2025-05-24 15:03:20 [INFO]: Epoch 015 - training

### Prédiction des valeurs

In [ ]:
data_test = pd.read_csv('data/processed/y_submit.csv')

In [4]:
saits_path = "saits/small_saits_X_test_sma.pypots/20250524_T144855/SAITS.pypots"
loaded_model = torch.load(saits_path, map_location=saits.device, weights_only=False)
type(loaded_model)

dict

In [5]:
loaded_model.keys()

dict_keys(['model_state_dict', 'pypots_version'])

In [2]:
saits_path = "saits/small_saits_X_test_sma.pypots/20250524_T144855/SAITS.pypots"
loaded_model = torch.load(saits_path, map_location=saits.device, weights_only=False)
saits.model.load_state_dict(loaded_model["model_state_dict"])

Y, scaler = preprocess_test(data_test)
Y_ori = Y.copy()
dataset = {"X": Y}

y_pred = saits.impute(dataset)

y_pred = postprocess_test(y_pred, scaler)

OutOfMemoryError: CUDA out of memory. Tried to allocate 8.32 GiB. GPU 0 has a total capacity of 21.95 GiB of which 4.81 GiB is free. Including non-PyTorch memory, this process has 17.13 GiB memory in use. Of the allocated memory 16.84 GiB is allocated by PyTorch, and 79.99 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [5]:
saits_path = "saits/small_saits_X_test_sma.pypots/20250524_T144855/SAITS.pypots"
loaded_model = torch.load(saits_path, map_location=saits.device, weights_only=False)
saits.model.load_state_dict(loaded_model["model_state_dict"])

Y, scaler = preprocess_test(data_test)
Y_ori = Y.copy()
dataset = {"X": Y}

import numpy as np

# Supposons que Y soit un array NumPy de forme (n_samples, n_steps, n_features)
# Y = ...
# scaler = ... (votre scaler)
# saits = ... (votre modèle SAITS entraîné)

chunk_size = 64 # Déterminez une taille de chunk gérable par votre GPU
y_pred_list = []

print(f"Shape de Y: {Y.shape}")

for i in range(0, Y.shape[0], chunk_size):
    print(f"Processing chunk {i // chunk_size + 1}...")
    Y_chunk = Y[i:i + chunk_size]
    dataset_chunk = {"X": Y_chunk}
    
    # Assurez-vous que le modèle ne garde pas d'état problématique entre les appels
    # ou qu'il est bien stateless pour l'imputation sur des chunks indépendants
    y_pred_chunk_raw = saits.impute(dataset_chunk)
    
    # Post-traitement si nécessaire sur le chunk
    # y_pred_chunk_processed = postprocess_test(y_pred_chunk_raw, scaler_relevant_pour_ce_chunk_ou_global)
    # Ici, on suppose que postprocess_test peut être appliqué après la concaténation
    
    y_pred_list.append(y_pred_chunk_raw)
    
    # Optionnel mais recommandé : libérer la mémoire GPU explicitement
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Concaténer les résultats (qui sont maintenant des arrays NumPy sur le CPU)
y_pred_full = np.concatenate(y_pred_list, axis=0)

# Appliquer le post-traitement global si ce n'est pas fait par chunk
y_pred = postprocess_test(y_pred_full, scaler)

print(f"Shape de y_pred final: {y_pred.shape}")

Shape de Y: (1000, 1057, 9)
Processing chunk 1...
Processing chunk 2...
Processing chunk 3...
Processing chunk 4...
Processing chunk 5...
Processing chunk 6...
Processing chunk 7...
Processing chunk 8...
Processing chunk 9...
Processing chunk 10...
Processing chunk 11...
Processing chunk 12...
Processing chunk 13...
Processing chunk 14...
Processing chunk 15...
Processing chunk 16...
Shape de y_pred final: (1057, 1000)


### Fonction pour évaluer le modèle

In [6]:
def MAE(y_true, y_pred, y_holed):
    mask = y_holed.isna()
    mae = np.abs(y_true[mask] - y_pred[mask]).mean().mean()
    return mae

holed_cols = [f"holed_{i}" for i in range(1, 1001)]
y_holed = data_test[holed_cols]
y_true = pd.read_csv('data/processed/y_train_complete.csv')
del y_true["Horodate"]

In [7]:
mae = MAE(y_true, y_pred, y_holed)

print(f"MAE: {mae}")

MAE: 105.45877137885287


In [11]:
import torch
import gc

# Supposons que vous ayez des tenseurs
# a = torch.randn(1000, 1000).cuda()
# b = torch.randn(1000, 1000).cuda()


# Forcer le garbage collector de Python
gc.collect()

# Vider le cache de mémoire allouée par PyTorch (très utile)
torch.cuda.empty_cache()

### Création du fichier submit

In [ ]:
y_pred.insert(0, "Horodate", data_test["Horodate"])
y_pred.to_csv('submit_sma.csv', index=False)

In [ ]:
y_pred.insert(0, "Horodate", data["Horodate"])
y_holed.insert(0, "Horodate", data["Horodate"])

### KNN Imputer

In [ ]:
X_test = pd.read_csv("data/X_test.csv")
y_test = pd.read_csv("data/y_test.csv")

del X_test["Horodate"]
#del y_true["Horodate"]
del y_test["Horodate"]

df = pd.concat([X_test, y_test], axis=1)

In [ ]:
df = pd.read_csv("original_data/X_train_78VdSWL.csv")
del df["Horodate"]

In [ ]:
df.head()

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
data_scaled = scaler.fit_transform(df)

imputer = KNNImputer(n_neighbors=4, weights='distance')
data_imputed = imputer.fit_transform(data_scaled)

# Inverse transform pour retrouver les valeurs d'origine
data_final = scaler.inverse_transform(data_imputed)
df = pd.DataFrame(data_final)
df = df.iloc[:, -1000:]
df.columns = y_true.columns

#mae = MAE(y_true, df, y_holed)

#print(f"MAE pour n = {i}: {mae}")

In [ ]:
df.insert(0, "Horodate", data["Horodate"])
df.to_csv('submit_knn_n=3.csv', index=False)

### Observation de la prédiction et de la valeur réel

In [ ]:
def plot_curve_comparison(real_curve, reconstructed_curve, index=0):
    """
    Trace la courbe réelle, la courbe bruitée et la courbe reconstruite pour un échantillon.
    
    Parameters:
        real_curve (numpy.array): Courbe réelle (1D, de longueur nb_timestamps)
        noisy_curve (numpy.array): Courbe bruitée avec données manquantes (1D, même longueur)
        reconstructed_curve (numpy.array): Courbe reconstruite (1D, même longueur)
        index (int): Index de l'échantillon (pour le titre du graphique)
    """
    timesteps = np.arange(len(real_curve))
    
    plt.figure(figsize=(12, 6))
    plt.plot(timesteps, real_curve, label='Courbe réelle', color='blue')
    plt.plot(timesteps, reconstructed_curve, label='Courbe reconstruite', color='red', linestyle='--')
    
    plt.xlabel('Timestamp')
    plt.ylabel('Valeur de consommation')
    plt.title(f'Comparaison des courbes (échantillon {index})')
    plt.legend()
    plt.grid(True)
    plt.show()

def compute_error_metrics(real_curves, reconstructed_curves):
    """
    Calcule les indicateurs d'erreur MAE et MSE entre les courbes réelles et reconstruites.
    
    Parameters:
        real_curves (numpy.array): Tableau de dimension (n_samples, n_timesteps)
        reconstructed_curves (numpy.array): Tableau de dimension (n_samples, n_timesteps)
    
    Returns:
        mae (numpy.array): MAE calculée pour chaque échantillon
        mse (numpy.array): MSE calculée pour chaque échantillon
    """
    mae = np.mean(np.abs(real_curves - reconstructed_curves), axis=1)
    mse = np.mean((real_curves - reconstructed_curves)**2, axis=1)
    return mae, mse

In [ ]:
real_curves = y_true.values.T
reconstructed_curves = y_pred.values.T

sample_index = random.randint(1,1000)
plot_curve_comparison(real_curves[sample_index], reconstructed_curves[sample_index], index=sample_index)

### Test MAE par taille

In [ ]:
from evaluation import mae_par_taille_trou

mae = mae_par_taille_trou(y_true, y_pred, y_holed)

mae

In [ ]:
from evaluation import mae_par_taille_trou

mae = mae_par_taille_trou(y_true, df, y_holed)

mae